# 07 Prophet

보상별 주간 등장 0/1 시계열에 Prophet을 적용해 다음 회차에 무엇이 나올지 정렬한다. 분류기와 같은 후보 universe, 같은 시간순 분할, 같은 `date_hit_rate`로 비교하고, Phase 16의 재증 게이트와 도메인 단종 목록을 그대로 재사용한다.

In [ ]:
from pathlib import Path
import logging
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

warnings.filterwarnings("ignore", message="IProgress not found.*")
logging.getLogger("cmdstanpy").disabled = True
logging.getLogger("prophet").disabled = True
logging.getLogger("prophet.plot").disabled = True

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.predict_dataset import DEFAULT_RECENCY_GATE_DAYS, DOMAIN_DISCONTINUED, _candidate_rewards, _eligible_rows, build_inference_frame, gate_candidates, write_prediction_frame
from src.prophet_cycle import build_weekly_series, fit_predict_reward, rank_candidates

RANDOM_SEED = 42
SPLIT_CUTOFF = pd.Timestamp("2025-01-01")

## 데이터와 후보

`data/processed/tidy_rewards.csv`와 Phase 16에서 만든 `data/interim/manual_obs_20260614.csv`를 사용한다. 원천 데이터는 읽지 않고 수정하지 않는다. Prophet 후보 universe는 분류기와 같은 학습 구간 빈도 상위 20개다.

In [ ]:
tidy_path = ROOT / "data" / "processed" / "tidy_rewards.csv"
manual_obs_path = ROOT / "data" / "interim" / "manual_obs_20260614.csv"
predict_path = ROOT / "data" / "interim" / "predict_dataset.csv"

tidy_raw = pd.read_csv(tidy_path)
manual_obs = pd.read_csv(manual_obs_path, parse_dates=["date"])
eligible = _eligible_rows(tidy_raw)
train_eligible = eligible.loc[eligible["date"] < SPLIT_CUTOFF]
candidates = _candidate_rewards(train_eligible, 20)
test_dates = pd.Index(sorted(eligible.loc[eligible["date"] >= SPLIT_CUTOFF, "date"].drop_duplicates()))
actual_by_date = {date: set(group["reward_norm"]) for date, group in eligible.groupby("date", sort=False)}
summary = {
    "후보수": len(candidates),
    "테스트회차": len(test_dates),
    "테스트시작일": test_dates.min().date(),
    "테스트종료일": test_dates.max().date(),
    "재증게이트일수": DEFAULT_RECENCY_GATE_DAYS,
    "도메인단종": sorted(DOMAIN_DISCONTINUED),
}
summary

## 분류기 기준표

분류기 기준표는 06_predict와 같은 코드 경로로 다시 계산한다. 모델과 하이퍼파라미터는 바꾸지 않는다.

In [ ]:
write_prediction_frame(tidy_path=tidy_path, output_path=predict_path, candidate_cutoff=SPLIT_CUTOFF)
df = pd.read_csv(predict_path, parse_dates=["date"])
train = df.loc[df["date"] < SPLIT_CUTOFF].copy()
test = df.loc[df["date"] >= SPLIT_CUTOFF].copy()
feature_columns = [
    "days_since_last",
    "past_count",
    "past_rate",
    "roll_gap_mean",
    "roll_gap_median",
    "lag1_appeared",
    "category",
    "month",
    "quarter",
    "weekofyear",
]

def evaluate(y_true, y_score, dates):
    y_true = pd.Series(y_true).astype(int).reset_index(drop=True)
    y_score = pd.Series(y_score).astype(float).reset_index(drop=True)
    dates = pd.Series(dates).reset_index(drop=True)
    y_pred = (y_score >= 0.5).astype(int)
    work = pd.DataFrame({"date": dates, "y": y_true, "score": y_score})

    def date_hit(group):
        k = int(group["y"].sum())
        if k < 1:
            return np.nan
        return float(group.nlargest(k, "score")["y"].max())

    auc = roc_auc_score(y_true, y_score) if y_true.nunique() > 1 else np.nan
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": auc,
        "date_hit_rate": work.groupby("date", sort=True).apply(date_hit).mean(),
    }

X_all = pd.get_dummies(df[feature_columns], columns=["category"], dummy_na=True).fillna(-1)
X_train = X_all.loc[train.index]
X_test = X_all.loc[test.index]
y_train = train["y"].astype(int)
y_test = test["y"].astype(int)

b1_score = (test["roll_gap_median"].notna() & (test["days_since_last"] >= test["roll_gap_median"])).astype(float)
b2_score = test["past_rate"].clip(0, 1)
logistic_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED)
tree_model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED)
logistic_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)
logistic_score = logistic_model.predict_proba(X_test)[:, 1]
tree_score = tree_model.predict_proba(X_test)[:, 1]
classifier_comparison = pd.DataFrame(
    [
        {"방법": "간격규칙", **evaluate(y_test, b1_score, test["date"])},
        {"방법": "사전확률", **evaluate(y_test, b2_score, test["date"])},
        {"방법": "로지스틱", **evaluate(y_test, logistic_score, test["date"])},
        {"방법": "랜덤포레스트", **evaluate(y_test, tree_score, test["date"])},
    ]
)
classifier_comparison

## Prophet 백테스트

Prophet은 보상별 주간 0/1 시계열을 학습하고, 28일 주기 시즈널리티와 연간 시즈널리티로 test 구간의 `yhat`을 예측한다. `yhat`은 보정된 확률이 아니라 상대 강도 점수다. 백테스트의 Prophet 적합은 `2025-01-01` 이전 이력만 사용하고, 각 test 날짜의 게이트는 그 날짜 미만 이력만 본다.

In [ ]:
target_dates = pd.DatetimeIndex(test_dates)
prophet_scores = {}
for reward in candidates:
    series = build_weekly_series(tidy_raw, reward, SPLIT_CUTOFF)
    forecast = fit_predict_reward(series, target_dates)
    prophet_scores[reward] = forecast.set_index("ds")["yhat"]

prophet_rows = []
for date in test_dates:
    gated = gate_candidates(
        eligible,
        candidates,
        date,
        recency_gate_days=DEFAULT_RECENCY_GATE_DAYS,
        domain_discontinued=DOMAIN_DISCONTINUED,
    )
    actual = actual_by_date.get(date, set())
    k = len(actual & set(candidates))
    if k < 1 or not gated:
        hit = np.nan
    else:
        ranked = sorted(gated, key=lambda reward: (-float(prophet_scores[reward].loc[date]), reward))
        hit = float(bool(set(ranked[:k]) & actual))
    prophet_rows.append({"date": date, "hit": hit})

prophet_date_hit_rate = pd.DataFrame(prophet_rows)["hit"].mean()
lookup = classifier_comparison.set_index("방법")["date_hit_rate"]
method_comparison = pd.DataFrame(
    [
        {"방법": "간격규칙", "date_hit_rate": float(lookup.loc["간격규칙"])},
        {"방법": "사전확률", "date_hit_rate": float(lookup.loc["사전확률"])},
        {"방법": "랜덤포레스트", "date_hit_rate": float(lookup.loc["랜덤포레스트"])},
        {"방법": "Prophet", "date_hit_rate": prophet_date_hit_rate},
    ]
)
method_comparison

## 2026-06-14 검증

06-14 실제 보상은 Phase 16의 보조 관측 파일을 사용한다. Prophet도 게이트 전에는 단종 보상이 올라올 수 있으므로, 게이트 전후를 모두 확인한다.

In [ ]:
actual_0614 = set(manual_obs["reward_norm"])

def verify_rank(table, actual_rewards, score_col):
    top5_hits = int(table.head(5)["reward_norm"].isin(actual_rewards).sum())
    rows = []
    for reward in sorted(actual_rewards):
        matched = table.loc[table["reward_norm"] == reward]
        if matched.empty:
            rows.append({"reward_norm": reward, "후보포함": False, "순위": pd.NA, score_col: pd.NA})
        else:
            row = matched.iloc[0]
            rows.append({"reward_norm": reward, "후보포함": True, "순위": int(row["rank"]), score_col: float(row[score_col])})
    return top5_hits, pd.DataFrame(rows)

def classifier_predict_for_date(tidy_source, target_date, gate):
    infer_frame = build_inference_frame(
        tidy_source,
        target_date,
        candidate_cutoff=SPLIT_CUTOFF,
        recency_gate_days=DEFAULT_RECENCY_GATE_DAYS if gate else None,
        domain_discontinued=DOMAIN_DISCONTINUED if gate else frozenset(),
    )
    X_target = pd.get_dummies(infer_frame[feature_columns], columns=["category"], dummy_na=True)
    X_target = X_target.reindex(columns=X_all.columns, fill_value=0).fillna(-1)
    table = infer_frame[["date", "reward_norm"]].copy()
    table["score_pct"] = (tree_model.predict_proba(X_target)[:, 1] * 100).round(1)
    table = table.sort_values(["score_pct", "reward_norm"], ascending=[False, True]).reset_index(drop=True)
    table["rank"] = np.arange(1, len(table) + 1)
    return table

prophet_0614_raw = rank_candidates(
    tidy_raw,
    candidates,
    as_of=pd.Timestamp("2026-06-14"),
    target_date=pd.Timestamp("2026-06-14"),
    recency_gate_days=None,
    domain_discontinued=frozenset(),
)
prophet_0614_gate = rank_candidates(
    tidy_raw,
    candidates,
    as_of=pd.Timestamp("2026-06-14"),
    target_date=pd.Timestamp("2026-06-14"),
    recency_gate_days=DEFAULT_RECENCY_GATE_DAYS,
    domain_discontinued=DOMAIN_DISCONTINUED,
)
prophet_0614_raw["yhat"] = prophet_0614_raw["yhat"].round(3)
prophet_0614_gate["yhat"] = prophet_0614_gate["yhat"].round(3)
classifier_0614_gate = classifier_predict_for_date(tidy_raw, pd.Timestamp("2026-06-14"), True)
prophet_hits, prophet_actual = verify_rank(prophet_0614_gate, actual_0614, "yhat")
classifier_hits, classifier_actual = verify_rank(classifier_0614_gate, actual_0614, "score_pct")
verification_0614 = pd.DataFrame(
    [
        {"방법": "랜덤포레스트", "상위5_적중수": classifier_hits, "후보수": len(classifier_0614_gate)},
        {"방법": "Prophet", "상위5_적중수": prophet_hits, "후보수": len(prophet_0614_gate)},
    ]
)
display(verification_0614)
display(classifier_actual.assign(**{"방법": "랜덤포레스트"})[["방법", "reward_norm", "후보포함", "순위", "score_pct"]])
display(prophet_actual.assign(**{"방법": "Prophet"})[["방법", "reward_norm", "후보포함", "순위", "yhat"]])
display(prophet_0614_raw.head(5))
display(prophet_0614_gate.head(5))

## 2026-06-21 예보 비교

06-21 예보는 06-14 보조 관측을 기준일 이전 이력으로 포함한다. 분류기는 확률 점수, Prophet은 `yhat` 상대 강도이므로 점수의 단위는 다르다. 같은 게이트를 적용한 상위 5개 순위만 나란히 비교한다.

In [ ]:
category_lookup = tidy_raw.dropna(subset=["reward_norm", "category"]).groupby("reward_norm")["category"].agg(lambda s: s.mode().iloc[0])
manual_for_tidy = manual_obs.copy()
manual_for_tidy["category"] = manual_for_tidy["reward_norm"].map(category_lookup)
manual_for_tidy["is_future"] = False
manual_for_tidy["no_event_week"] = False
tidy_with_manual = pd.concat([tidy_raw, manual_for_tidy], ignore_index=True, sort=False)

classifier_0621 = classifier_predict_for_date(tidy_with_manual, pd.Timestamp("2026-06-21"), True)
prophet_0621 = rank_candidates(
    tidy_with_manual,
    candidates,
    as_of=pd.Timestamp("2026-06-21"),
    target_date=pd.Timestamp("2026-06-21"),
    recency_gate_days=DEFAULT_RECENCY_GATE_DAYS,
    domain_discontinued=DOMAIN_DISCONTINUED,
)
prophet_0621["yhat"] = prophet_0621["yhat"].round(3)
excluded_0621 = {
    "현상금_경험치": "현상금_경험치" not in set(prophet_0621["reward_norm"]) and "현상금_경험치" not in set(classifier_0621["reward_norm"]),
    "불꽃늑대_경험치": "불꽃늑대_경험치" not in set(prophet_0621["reward_norm"]) and "불꽃늑대_경험치" not in set(classifier_0621["reward_norm"]),
    "스타포스_1+1강화": "스타포스_1+1강화" not in set(prophet_0621["reward_norm"]) and "스타포스_1+1강화" not in set(classifier_0621["reward_norm"]),
    "스타포스_성공확률": "스타포스_성공확률" not in set(prophet_0621["reward_norm"]) and "스타포스_성공확률" not in set(classifier_0621["reward_norm"]),
}
assert all(excluded_0621.values())
comparison_0621 = pd.DataFrame(
    {
        "순위": range(1, 6),
        "랜덤포레스트_보상": classifier_0621.head(5)["reward_norm"].to_list(),
        "랜덤포레스트_확률(%)": classifier_0621.head(5)["score_pct"].to_list(),
        "Prophet_보상": prophet_0621.head(5)["reward_norm"].to_list(),
        "Prophet_yhat": prophet_0621.head(5)["yhat"].to_list(),
    }
)
display(comparison_0621)
excluded_0621

## 결론 프레임

Prophet은 보상별 재등장 시계열의 28일 주기와 연간 변동을 직접 모델링한다. 이 방식은 몬스터파크처럼 분류기가 낮게 둔 보상을 더 높게 볼 수 있지만, 강한 과거 주기를 가진 단종 보상도 계속 추천할 수 있다. 따라서 Prophet도 분류기와 마찬가지로 재증 게이트와 도메인 단종 목록이 필요하다. 이번 비교의 결론은 단일 승자가 아니라 두 방식이 서로 다른 신호를 보며, 둘 다 운영상 사망한 보상을 별도로 걸러야 한다는 점이다.